# Phase 1 — Collect Opinions & Run Mediation

**What this does:** Takes free-text opinions from a Google Form export and runs
the deliberation loop using Claude to produce a small set of refined consensus
statements. At the end it prints a ready-made template for building your voting form.

---

## Before you start: set up the opinions form

Create a **Google Form** (or Microsoft Form) with exactly two questions:

| # | Type | Question text |
|---|------|---------------|
| 1 | Short answer | `Your name` |
| 2 | Paragraph | `What matters most to you about [your topic]? What would make it work — or not work — for you?` |

Once responses are in:
- Google Forms → Responses tab → ⋮ → **Download responses (.csv)**
- Microsoft Forms → Responses tab → **Open in Excel** → Save as CSV

Save the CSV in this `notebooks/` folder, then update `OPINIONS_CSV` below.

In [ ]:
# ── CONFIG — edit these values ───────────────────────────────────────────────
TOPIC        = "Where should we hold the combined summer social for engineering and marketing?"
OPINIONS_CSV = "example_opinions.csv"   # ← your Google/Microsoft Form CSV export

NUM_CANDIDATES = 6    # candidate statements generated per round (5–8 works well)
MAX_ROUNDS     = 2    # deliberation rounds — 1 is often enough; 2 refines further
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys, os, json, asyncio
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()   # lets asyncio.run() work inside Jupyter's existing event loop

# Make sure the project root is on the path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = Path("session_data")
OUTPUT_DIR.mkdir(exist_ok=True)

api_key = os.getenv("ANTHROPIC_API_KEY", "")
print(f"Topic      : {TOPIC}")
print(f"API key    : {'✓ loaded' if api_key else '⚠  NOT FOUND — add ANTHROPIC_API_KEY to ../.env'}")
print(f"Opinions   : {OPINIONS_CSV}")
print(f"Output dir : {OUTPUT_DIR.resolve()}")

## Step 1 — Load and preview opinions

In [ ]:
import pandas as pd

df = pd.read_csv(OPINIONS_CSV)
print(f"{len(df)} responses loaded")
print(f"Columns: {list(df.columns)}\n")
df

In [ ]:
# ── Column mapping ────────────────────────────────────────────────────────────
# Google Forms: col 0 = Timestamp, col 1 = name, col 2 = opinion
# Microsoft Forms: col 0 = ID, col 1 = Start time, ... adjust if different
NAME_COL    = df.columns[1]   # ← change index if needed
OPINION_COL = df.columns[2]   # ← change index if needed

print(f"Name column    : '{NAME_COL}'")
print(f"Opinion column : '{OPINION_COL}'\n")

for _, row in df.iterrows():
    name = str(row[NAME_COL]).strip()
    text = str(row[OPINION_COL]).strip()
    snippet = text[:100] + ("..." if len(text) > 100 else "")
    print(f"  {name:<20} {snippet}")

## Step 2 — Build participant and opinion objects

In [ ]:
from group_consensus.models.types import Opinion, Participant, SessionConfig
from group_consensus.mediation.async_mediator import AsyncMediator

participants = []
opinions     = []
SESSION_ID   = "session_01"

for i, row in df.iterrows():
    pid  = f"p{i}"
    name = str(row[NAME_COL]).strip()
    text = str(row[OPINION_COL]).strip()
    participants.append(Participant(id=pid, name=name))
    opinions.append(Opinion(participant_id=pid, text=text, session_id=SESSION_ID))

config = SessionConfig(
    session_id=SESSION_ID,
    topic=TOPIC,
    num_candidate_statements=NUM_CANDIDATES,
    max_deliberation_rounds=MAX_ROUNDS,
)
mediator = AsyncMediator(config)

print(f"✓ {len(participants)} participants ready")
print(f"  Running {MAX_ROUNDS} round(s), {NUM_CANDIDATES} candidates each")
print(f"  Concurrent API calls — should take ~20–40 seconds total")

## Step 3 — Run the mediation loop

This makes concurrent API calls to Claude — one batch per round.
With 15 participants and 2 rounds expect **30–60 seconds**.

In [ ]:
result = asyncio.run(
    mediator.run(
        topic=TOPIC,
        participants=participants,
        opinions=opinions,
        verbose=True,
    )
)

## Step 4 — Review all candidates and the winner

In [ ]:
for rnd in result.rounds:
    print(f"\n{'═'*55}")
    print(f"  Round {rnd.round_number + 1} — all {len(rnd.candidate_statements)} candidates")
    print(f"{'═'*55}")
    for i, stmt in enumerate(rnd.candidate_statements, 1):
        marker = "  ← Schulze winner" if stmt.id == rnd.winning_statement.id else ""
        print(f"  {i}. {stmt.text}{marker}")

## Step 5 — Save outputs and print the voting form template

The final-round candidates become the statements people will vote on.
The template below gives you the copy-paste text for building your voting form.

In [ ]:
# Collect all statements from all rounds (for the analysis notebook)
all_stmts = []
seen_ids  = set()
for rnd in result.rounds:
    for s in rnd.candidate_statements:
        if s.id not in seen_ids:
            all_stmts.append(s)
            seen_ids.add(s.id)

# Save statements.json
session_data = {
    "session_id"            : config.session_id,
    "topic"                 : TOPIC,
    "consensus_statement_id": str(result.consensus_statement.id),
    "statements"            : [
        {"id": str(s.id), "text": s.text, "type": s.type, "round": s.round_number}
        for s in all_stmts
    ],
}
with open(OUTPUT_DIR / "statements.json", "w") as f:
    json.dump(session_data, f, indent=2)

# The voting form uses only the final round's candidates (avoids overwhelming voters)
voting_stmts = result.rounds[-1].candidate_statements

# Save label→statement map so 02_analyse.ipynb can match column names to IDs
label_map = {
    f"Statement {i}": {"id": str(s.id), "text": s.text}
    for i, s in enumerate(voting_stmts, 1)
}
with open(OUTPUT_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

print(f"✓ Saved {len(all_stmts)} statements  →  session_data/statements.json")
print(f"✓ Saved label map              →  session_data/label_map.json")
print(f"  ({len(voting_stmts)} statements will appear in the voting form)")

In [ ]:
# Print the voting form template
DIVIDER = "═" * 60
print(DIVIDER)
print("  VOTING FORM TEMPLATE")
print("  Copy these into a new Google Form (or Microsoft Form)")
print(DIVIDER)
print()
print(f"Form title       : {TOPIC}")
print( "Form description : Please rate each statement. There are no right or wrong")
print( "                   answers — we want your honest reaction.")
print()
print("Add these questions (type = Multiple choice, options = Agree / Pass / Disagree):")
print()
for label, info in label_map.items():
    print(f"  Question title : {label}")
    print(f"  Description    : {info['text']}")
    print(f"  Options        : Agree  |  Pass  |  Disagree")
    print()
print("Also add a short-answer question: 'Your name'")
print()
print(DIVIDER)
print("Once responses are in, export as CSV and run 02_analyse.ipynb")
print(DIVIDER)